# Module 3 — NGS Pipeline with Claude Code (Alignment)

**SBML Lab Intern Training | KAIST**

---

This module covers the full NGS alignment pipeline — from raw SRA data to a sorted, indexed BAM and a GFF file ready for MetaScope. You'll use Claude Code to generate the pipeline commands, but you must understand what each step does before running it.

**Lab context:** Reference genome is *E. coli* K-12 MG1655. The dataset is **single-end ChIP-exo reads** — each read comes from one end of the DNA fragment, not a pair. Aligner is `bowtie2`. Post-alignment processing uses `samtools`, followed by `makegff.py` to produce a GFF file for MetaScope visualization.

**Learning objectives:**
- Understand how `bowtie2` aligns short reads to a reference genome
- Understand why BAM files must be sorted and indexed before downstream use
- Practice using Claude Code's **plan mode** for multi-step pipelines
- Annotate and verify every flag in a generated command before running it

## 0. File Formats — Know Before You Align

This pipeline produces and consumes several file formats. Use Claude Code to answer the questions below **before** starting the exercises.

| Format | Question to answer with Claude Code |
|--------|-------------------------------------|
| FASTQ  | What do the 4 lines per read represent? What does the quality score number mean? |
| FASTA  | How does it differ from FASTQ? When would you use one vs the other? |
| SAM    | How many columns are there? What does the FLAG field encode? |
| BAM    | Why is BAM preferred over SAM for storage and downstream tools? |
| GFF    | How many columns? What coordinate system (0-based or 1-based)? |

Write your answers below — one sentence per format.


> **Answer**
>
> *(Write your one-sentence answers here.)*


## 1. How `bowtie2` Works

`bowtie2` aligns short reads to a reference genome using an index built with `bowtie2-build`. Use Claude Code to understand how the aligner works before you run it.

### Exercise 1 — Understand bowtie2 Before Running It

Use Claude Code to understand how `bowtie2` finds alignments and what the key flags control. Write a 3-sentence summary in your own words in the cell below.

> **Answer**
>
> *(Write your 3-sentence summary here.)*


## 2. `samtools` Internals

`samtools` converts, sorts, and indexes alignment files. Use Claude Code to understand the difference between SAM and BAM, and why the sort step must come before the index step.

### Exercise 2 — Sort Before Index

**Without asking Claude Code first:** write in the cell below why you must sort a BAM file before indexing it. Use your own reasoning.

Then verify your explanation with Claude Code.

> **Answer**
>
> *(Write your reasoning here before consulting Claude Code.)*


## 3. Plan Mode — Reviewing Pipelines Before Running

Claude Code's **plan mode** lets you see every step Claude intends to take before a single command is executed.

**How to activate:** Press **Shift+Tab** before sending your prompt. The interface switches to plan mode. Claude will show you the full sequence of actions — file reads, commands, edits — and wait for your approval.

**Why this matters for pipelines:**  
A multi-step NGS pipeline touches raw data, builds index files, and writes intermediate outputs. A mistake at step 2 (wrong index path) silently produces a valid-looking but incorrect BAM. Reviewing the plan lets you:
- Catch wrong paths or flag values before they cause silent errors
- Understand every command before it runs
- Modify individual steps (e.g., add `--no-unal`, increase `-p` threads) without re-generating the whole plan

**Rule for this module:** Any time you ask Claude Code to generate a multi-step pipeline, you must use plan mode and review it before approving.

### Exercise 3 — Plan Mode Pipeline Review

Use plan mode (Shift+Tab) to generate the full pipeline:
```
FASTQ → bowtie2 align → samtools sort → samtools index → makegff.py
```
Review every proposed step before approving. Identify at least one flag or parameter you would change or verify, and explain why.

> **Answer**
>
> *(Write the flag/parameter you identified, your change, and your reasoning here.)*


## 4. SRA Download with `fastq-dump`

`fastq-dump` downloads sequencing data from NCBI's Sequence Read Archive by accession number. Use Claude Code to generate the correct command with the right flags for your dataset.

In [ ]:
%%bash
# Exercise 4 — Generate the fastq-dump command for a given SRR accession using Claude Code.
# Paste the command here and annotate each flag with a comment.


## 5. SAM FLAG Values

Every SAM record has a FLAG field in column 2 — a bitwise integer encoding alignment properties. Use Claude Code to understand what FLAG values mean and how to filter by them with `samtools view`.

### Exercise 5 — Decode an Unknown FLAG

> Look up what a specific FLAG value means using Claude Code. Choose a value you have **not** looked up yet (e.g., 2048, 256, 1024).
>
> Write:
> 1. The FLAG value you chose
> 2. What it means (decoded bits)
> 3. How you would verify that a real SAM record carries this flag (e.g., with `samtools view -f` or `samtools flags`)

> **Answer**
>
> *(Write your FLAG value, its meaning, and verification method here.)*


## 6. Writing `makegff.py`

`makegff.py` is the final step in the pipeline — it converts your sorted BAM into a GFF file that MetaScope can load. This script is not pre-provided. Use Claude Code to write it before running the full pipeline. Consult `lab-context.md` for the GFF format this lab uses.

### Exercise 6 — Write `makegff.py`

Use Claude Code to write a Python script named `makegff.py` that:
1. Takes a sorted BAM file as input
2. Outputs a GFF-format file readable by MetaScope

Test it on your aligned BAM before running the full pipeline in Exercise 7.

Write the final working script in the code cell below.

**Expected output format** (one row per aligned read):
```
NC_000913.3	makegff	read	12345	12396	1	+	.	name=SRR000000.1
```
Columns: seqname, source (`makegff`), feature (`read`), start, end, score (value: `1` per read), strand, `.`, read name.
Coordinates are **1-based**. Each aligned read becomes one GFF row.


In [ ]:
# makegff.py — your script here


## 7. Running the Full Pipeline on Sample Data

Download the reference genome and reads from NCBI following `data/reference/README.md`, then use plan mode to generate the full pipeline. Fill in each command below after reviewing the plan.

In [ ]:
%%bash
# Full NGS pipeline — fill in the commands after generating them with Claude Code in plan mode.
# Do NOT run this cell until you have reviewed the plan and annotated each flag.

# Step 1: Build bowtie2 index from reference FASTA


# Step 2: Align single-end reads with bowtie2


# Step 3: Convert SAM to BAM (samtools view)


# Step 4: Sort BAM by coordinate (samtools sort)


# Step 5: Index sorted BAM (samtools index)


# Step 6: Run your makegff.py to generate GFF for MetaScope


## End of Session

Before closing:

1. Make sure your pipeline cell runs end-to-end without errors.
2. Verify that the output BAM is coordinate-sorted: `samtools view -H output.bam | grep SO`
3. Verify the index exists: `ls -lh output.bam.bai`
4. Verify your GFF output has content: `wc -l output.gff`
5. Run `/log` in Claude Code to save a session log.

---

**Run `/log` before closing.**